# Nemotron Offline Prompt Eval — 2026-04-27

Kaggle-safe offline prompt evaluation notebook for the NVIDIA Nemotron Model Reasoning Challenge.

Purpose:
- avoid internet-dependent installs
- avoid TRL / SFTTrainer
- avoid LoRA training in the current constrained environment
- compare prompt formats on a small staged sample
- align with the competition metric, which prioritizes final answers inside \boxed{}


In [1]:
import os
import sys
import gc
import re
import json
import time
import random
import subprocess
import site
from pathlib import Path

# -----------------------------
# 0) Resolve NVIDIA utility script path
# -----------------------------
candidate_paths = [
    '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script',
    '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script',
    '/kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script',
]

UTILITY_SCRIPT_PATH = None
for p in candidate_paths:
    if Path(p).exists():
        UTILITY_SCRIPT_PATH = p
        break

print('Resolved UTILITY_SCRIPT_PATH:', UTILITY_SCRIPT_PATH)
if UTILITY_SCRIPT_PATH is None:
    raise FileNotFoundError(
        'Could not locate NVIDIA utility script path. '
        'Expected one of: ' + ', '.join(candidate_paths)
    )

# -----------------------------
# 1) Nemotron runtime prep
# -----------------------------
commands = [
    'uv pip uninstall torch torchvision torchaudio',
    f'tar -cf - -C {UTILITY_SCRIPT_PATH} . | tar -xf - -C /tmp',
    'chmod +x /tmp/triton/backends/nvidia/bin/ptxas',
    'chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell',
]
for cmd in commands:
    print(f'Running: {cmd}')
    subprocess.run(cmd, shell=True, check=True)

sys.path.insert(0, '/tmp')
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TRITON_PTXAS_PATH'] = '/tmp/triton/backends/nvidia/bin/ptxas'
os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = '/tmp/triton/backends/nvidia/bin/ptxas-blackwell'

cutlass_candidates = [
    f'{UTILITY_SCRIPT_PATH}/nvidia_cutlass_dsl/python_packages',
    '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_cutlass_dsl/python_packages',
]
for c in cutlass_candidates:
    if Path(c).exists():
        site.addsitedir(c)
        print('Added CUTLASS python package dir:', c)

import polars as pl
import pandas as pd
import torch
import kagglehub
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_PATH = kagglehub.model_download('metric/nemotron-3-nano-30b-a3b-bf16/transformers/default')
TRAIN_PATH = '/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv'

print('MODEL_PATH:', MODEL_PATH)

train = pl.read_csv(TRAIN_PATH)
print('Full train shape:', train.shape)
print(train.head())

# Stage 1 sample is intentionally small to control quota.
EVAL_ROWS = 20
train_eval = train.sample(n=min(EVAL_ROWS, train.height), seed=SEED, shuffle=True)
print('Eval subset shape:', train_eval.shape)

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch_dtype,
    trust_remote_code=True,
    device_map='auto',
)
model.eval()

patched = False
for name, mod in list(sys.modules.items()):
    if 'modeling_nemotron_h' in name and hasattr(mod, 'is_fast_path_available'):
        mod.is_fast_path_available = False
        patched = True
        print(f'Disabled fast path in module: {name}')
print('Fast path disabled:', patched)
print('Model/tokenizer ready.')


Resolved UTILITY_SCRIPT_PATH: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script
Running: uv pip uninstall torch torchvision torchaudio


Using Python 3.12.12 environment at: /usr
Uninstalled 3 packages in 757ms
 - torch==2.10.0+cu128
 - torchaudio==2.10.0+cu128
 - torchvision==0.25.0+cu128


Running: tar -cf - -C /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script . | tar -xf - -C /tmp
Running: chmod +x /tmp/triton/backends/nvidia/bin/ptxas
Running: chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell
Added CUTLASS python package dir: /kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/nvidia_cutlass_dsl/python_packages
MODEL_PATH: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
Full train shape: (9500, 3)
shape: (5, 3)
┌──────────┬─────────────────────────────────┬───────────────────────┐
│ id       ┆ prompt                          ┆ answer                │
│ ---      ┆ ---                             ┆ ---                   │
│ str      ┆ str                             ┆ str                   │
╞══════════╪═════════════════════════════════╪═══════════════════════╡
│ 00066667 ┆ In Alice's Wonderland, a secre… ┆ 10010111              │
│ 000b53cf ┆ In Alice's Wonderland, a secre… ┆ 01000011              │
│ 00189f6a ┆

/tmp/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Disabled fast path in module: transformers_modules._1.modeling_nemotron_h
Fast path disabled: True
Model/tokenizer ready.


In [2]:
def extract_final_answer(text):
    if text is None:
        return 'NOT_FOUND'
    boxed = re.findall(r'\\boxed\{([^}]*)(?:\}|$)', text)
    if boxed:
        non_empty = [m.strip() for m in boxed if m.strip()]
        if non_empty:
            return non_empty[-1]
        return boxed[-1].strip()
    patterns = [
        r'The final answer is:\s*([^\n]+)',
        r'Final answer is:\s*([^\n]+)',
        r'Final answer\s*[:：]\s*([^\n]+)',
        r'final answer\s*[:：]\s*([^\n]+)',
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return matches[-1].strip()
    nums = re.findall(r'-?\d+(?:\.\d+)?', text)
    if nums:
        return nums[-1]
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return lines[-1] if lines else 'NOT_FOUND'

def is_correct(gt, pred):
    return str(gt).strip().lower() == str(pred).strip().lower()

PROMPT_VARIANTS = {
    'P0_baseline_boxed': '\nPlease put your final answer inside \\boxed{}.',
    'P1_single_boxed_only': '\nSolve carefully.\nPut your final answer inside \\boxed{}.\nReturn exactly one final boxed answer and nothing after it.',
    'P2_short_answer_strict': '\nSolve internally.\nIn the final response, output only one short final answer in \\boxed{}.\nDo not add explanation after the boxed answer.',
    'P3_metric_aligned': '\nPlease put your final answer inside \\boxed{}.\nFor example: \\boxed{your answer}\nReturn only one final boxed answer.',
}

MAX_NEW_TOKENS = 96

@torch.no_grad()
def generate_one(user_prompt, max_new_tokens=MAX_NEW_TOKENS):
    try:
        prompt_text = tokenizer.apply_chat_template(
            [{'role': 'user', 'content': user_prompt}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except Exception:
        prompt_text = user_prompt

    inputs = tokenizer(prompt_text, return_tensors='pt', truncation=True, max_length=4096)
    model_device = next(model.parameters()).device
    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=False,
    )

    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

rows = train_eval.to_dicts()
all_results = []

for variant_name, suffix in PROMPT_VARIANTS.items():
    print(f'\nRunning variant: {variant_name}')
    correct = 0
    start = time.time()

    for row in tqdm(rows):
        full_prompt = row['prompt'] + suffix
        raw_output = generate_one(full_prompt)
        pred = extract_final_answer(raw_output)
        gt = row['answer']
        match = is_correct(gt, pred)
        correct += int(match)
        all_results.append({
            'id': row['id'],
            'variant': variant_name,
            'ground_truth': gt,
            'prediction': pred,
            'correct': match,
            'raw_output': raw_output,
        })

    elapsed = time.time() - start
    accuracy = correct / len(rows)
    print(f'{variant_name} accuracy: {accuracy:.4f}')
    print(f'{variant_name} elapsed_sec: {elapsed:.1f}')

results_df = pd.DataFrame(all_results)
summary_df = (
    results_df.groupby('variant', as_index=False)['correct']
    .mean()
    .rename(columns={'correct': 'accuracy'})
    .sort_values('accuracy', ascending=False)
)

print('\nPrompt variant summary:')
print(summary_df)

results_df.to_csv('/kaggle/working/prompt_eval_detailed.csv', index=False)
summary_df.to_csv('/kaggle/working/prompt_eval_summary.csv', index=False)
print('Saved CSVs to /kaggle/working')



Running variant: P0_baseline_boxed


  0%|          | 0/20 [00:00<?, ?it/s]

P0_baseline_boxed accuracy: 0.1000
P0_baseline_boxed elapsed_sec: 196.2

Running variant: P1_single_boxed_only


  0%|          | 0/20 [00:00<?, ?it/s]

P1_single_boxed_only accuracy: 0.1000
P1_single_boxed_only elapsed_sec: 73.5

Running variant: P2_short_answer_strict


  0%|          | 0/20 [00:00<?, ?it/s]

P2_short_answer_strict accuracy: 0.1500
P2_short_answer_strict elapsed_sec: 73.4

Running variant: P3_metric_aligned


  0%|          | 0/20 [00:00<?, ?it/s]

P3_metric_aligned accuracy: 0.1500
P3_metric_aligned elapsed_sec: 75.3

Prompt variant summary:
                  variant  accuracy
3       P3_metric_aligned      0.15
2  P2_short_answer_strict      0.15
1    P1_single_boxed_only      0.10
0       P0_baseline_boxed      0.10
Saved CSVs to /kaggle/working


In [3]:
best_variant = summary_df.iloc[0]['variant']
print('Best variant:', best_variant)

best_df = results_df[results_df['variant'] == best_variant].copy()
wrong_df = best_df[best_df['correct'] == False].copy()

def classify_error(gt, pred, raw):
    gt_s = str(gt).strip()
    pred_s = str(pred).strip()
    raw_s = str(raw)
    if pred_s == 'NOT_FOUND':
        return 'not_found'
    if '\\boxed{' not in raw_s:
        return 'missing_box'
    if pred_s.lower() == gt_s.lower():
        return 'correct'
    if gt_s.isdigit() or re.fullmatch(r'[01]+', gt_s):
        return 'wrong_symbolic_or_numeric'
    return 'wrong_text'

if len(wrong_df) > 0:
    wrong_df['error_cluster'] = wrong_df.apply(
        lambda r: classify_error(r['ground_truth'], r['prediction'], r['raw_output']),
        axis=1,
    )
    cluster_df = (
        wrong_df.groupby('error_cluster', as_index=False)
        .size()
        .sort_values('size', ascending=False)
    )
    print('\nError clusters for best variant:')
    print(cluster_df)
    cluster_df.to_csv('/kaggle/working/prompt_eval_error_clusters.csv', index=False)
else:
    print('No wrong cases to cluster.')

print('\nSample wrong cases:')
print(wrong_df[['id', 'ground_truth', 'prediction']].head(10))

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('Done.')


Best variant: P3_metric_aligned

Error clusters for best variant:
               error_cluster  size
1                 wrong_text    11
0  wrong_symbolic_or_numeric     6

Sample wrong cases:
          id             ground_truth                      prediction
60  cf447906                 10000111                        11000110
62  fb8ae661                    32.84                           32.60
63  b275443b                 00111000                        11111111
64  c9e990de                     3.26                            1.02
65  964c8468                   100.74                           17.28
67  9b1761fb                 10000000                        00000000
68  34d1d16f                      ^|!                             @|[
69  ee61df9f                 01000100                        10001000
70  8f16da79                    16.53                           16.57
72  ad3558db  the ancient king writes  the rabbit jumps over the moon
Done.
